<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_2_Feature_Scaling/Day_14_Week_2_Review_and_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 14: Week 2 Review and Scaling & Transformation Project

## Phase 2 | Month 5 | Week 2 | Day 14

**🎯 Goal:** Consolidate Week 2 skills through a complete preprocessing pipeline project.
**⏱️ Study Session:** ~2-3 hours

### Week 2 Topics Covered
1. ✓ Introduction to Feature Scaling — why it matters, which models need it
2. ✓ StandardScaler and MinMaxScaler
3. ✓ RobustScaler and Normalisation
4. ✓ Log, Box-Cox, and Power Transformations
5. ✓ Binning and Discretisation
6. ✓ Feature Engineering Basics

## 📊 Mini-Project: Equity Bank Loan Preprocessing Pipeline

Build a full preprocessing pipeline that takes a messy loan application dataset and produces analysis-ready features.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
N = 2000

# Messy loan dataset
df = pd.DataFrame({
    'income_kes':   np.where(np.random.rand(N)<0.15, np.nan,
                             np.random.lognormal(10.8, 0.9, N)),
    'loan_amount':  np.random.lognormal(11.5, 0.7, N),
    'age':          np.where(np.random.rand(N)<0.08, np.nan,
                             np.random.normal(35, 12, N).clip(18, 75)),
    'years_employed': np.where(np.random.rand(N)<0.12, np.nan,
                               np.random.uniform(0, 35, N)),
    'num_prior_loans': np.random.poisson(1.5, N),
    'defaulted':    None,
})
# Generate target
logit = (-2 + 0.000005*(df['loan_amount']/(df['income_kes'].fillna(50000)))
         - 0.01*(df['years_employed'].fillna(5))
         + 0.02*(df['num_prior_loans']))
df['defaulted'] = (np.random.rand(N) < 1/(1+np.exp(-logit))).astype(int)

print(df.describe().round(0))
print(f'\nDefault rate: {df.defaulted.mean():.3f}')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
N = 2000
df = pd.DataFrame({
    'income_kes':     np.where(np.random.rand(N)<0.15, np.nan, np.random.lognormal(10.8,0.9,N)),
    'loan_amount':    np.random.lognormal(11.5,0.7,N),
    'age':            np.where(np.random.rand(N)<0.08, np.nan, np.random.normal(35,12,N).clip(18,75)),
    'years_employed': np.where(np.random.rand(N)<0.12, np.nan, np.random.uniform(0,35,N)),
    'num_prior_loans':np.random.poisson(1.5,N),
})
logit = -2 + 0.000005*(df['loan_amount']/(df['income_kes'].fillna(50000))) - 0.01*(df['years_employed'].fillna(5)) + 0.02*df['num_prior_loans']
y = (np.random.rand(N) < 1/(1+np.exp(-logit))).astype(int)

# Complete preprocessing pipeline
skewed_features = ['income_kes','loan_amount']
normal_features = ['age','years_employed','num_prior_loans']

skewed_pipeline = Pipeline([
    ('impute',    SimpleImputer(strategy='median')),
    ('transform', PowerTransformer(method='yeo-johnson')),
    ('scale',     StandardScaler()),
])

normal_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('skewed', skewed_pipeline, skewed_features),
    ('normal', normal_pipeline, normal_features),
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression()),
])

X = df[skewed_features + normal_features]
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 📋 Week 2 Summary

| Task | Tool |
|------|------|
| Z-score scaling | `StandardScaler` |
| Range [0,1] scaling | `MinMaxScaler` |
| Outlier-robust scaling | `RobustScaler` |
| Remove skewness | `PowerTransformer(Yeo-Johnson)` |
| Create bins | `pd.cut`, `pd.qcut`, `KBinsDiscretizer` |
| Feature combinations | `PolynomialFeatures`, manual ratios |
| Full pipeline | `Pipeline` + `ColumnTransformer` |

## 🚀 Next Steps
**Week 3: Encoding Categorical Variables** — convert text categories into numbers.

### 🌙 Weekend Challenge
Apply the full Equity Bank pipeline to a real dataset from Kenya Open Data. Add at least 3 engineered features and report AUC with and without them.